# 03 — DL 모형 (1D-CNN / LSTM / TFT / Mamba)

| 모델   | seq_len | 비고              |
|--------|---------|-------------------|
| 1DCNN  | 30      | short             |
| LSTM   | 60      | medium, 양방향    |
| TFT    | 60      | medium, 경량      |
| Mamba  | 120     | long, parallel scan |

# **산출물** (ML과 동일 구조):
- `results/dl_results_{tier}.csv`
- `results/dl_tuned_params.csv`
- `results/best_models/{model}/{regime}_{country}_{tier}.pkl`
- `log/{model}/log_{model}_{regime}_{country}.csv`

## 0. 환경 설정

In [ ]:
import os, sys
from pathlib import Path

# Colab 마운트
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.environ['FINTEL_PROJECT_ROOT'] = (
        '/content/drive/MyDrive/대학교/학부4학년_추가/'
        'datasciencecapstone/Project'
    )
except ImportError:
    pass

PROJECT_ROOT = Path(os.environ.get('FINTEL_PROJECT_ROOT', Path.cwd().parent))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# 패키지 설치 (Colab 최초 1회)
import subprocess
subprocess.run(
    ['pip', 'install', '-q', '-r',
     str(PROJECT_ROOT / 'requirements.txt')],
    check=True,
)

import time, warnings, random
from itertools import product
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import joblib

warnings.filterwarnings('ignore')

# Seed 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 프로젝트 모듈
from src import config
from src.data_loader import load_dataset
from src.preprocess import get_feature_list
from src.models import (
    DL_MODEL_REGISTRY, DL_SEQ_LEN,
    ALL_DL_MODEL_NAMES, make_dl_model,
)
from src.eval import (
    evaluate, iter_phases,
    tune_dl_model, run_dl_static, run_dl_expanding,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'Models  : {ALL_DL_MODEL_NAMES}')
print(f'Seq len : {DL_SEQ_LEN}')

## 1. 그리드 정의

In [ ]:
TIERS       = ['core', 'momentum', 'extended']
PROTOCOLS   = ['static', 'expanding']
REFIT_EVERY = 5
DL_N_TRIALS = 20

# 학습 설정
TRAIN_CFG = dict(max_epochs=100, patience=10, batch_size=64)
# expanding refit: 시간 절약 (성능 손실 최소화)
EXPAND_CFG = dict(max_epochs=30,  patience=5,  batch_size=64)

# 디버그용 override (None = 정상 실행)
N_TRIALS_OVERRIDE: int | None = None

print(f'regimes  : {config.REGIMES}')
print(f'countries: {config.COUNTRIES}')
print(f'tiers    : {TIERS}')
print(f'protocols: {PROTOCOLS}')
print(f'n_trials : {DL_N_TRIALS}')

## 2. 데이터 로딩 헬퍼

In [ ]:
def load_split_3way(regime: str, country: str):
    """CSV split 컬럼 기반 3분할 — ML(02)과 동일."""
    df = load_dataset(regime, country)
    df = df.dropna(subset=['RV_target', 'RV_d', 'RV_w', 'RV_m', 'log_return'])
    return (
        df[df['split'] == 'train'].copy(),
        df[df['split'] == 'valid'].copy(),
        df[df['split'] == 'test'].copy(),
    )

# Sanity check
tr, va, te = load_split_3way('normal', 'US')
print(f'normal/US  train={len(tr)} valid={len(va)} test={len(te)}')
print(f'Mamba 최소 필요 길이: seq_len={DL_SEQ_LEN["Mamba"]} → train 충분: {len(tr) > DL_SEQ_LEN["Mamba"]}')


## 3. 그리드 실행

### 각 (regime × country × tier × model) 조합:
1. Optuna 튜닝 (train + valid → best_params)
2. static + expanding 프로토콜로 test 예측
3. phase별 평가 → 결과 행 누적

### **예상 소요 시간** (T4 기준):
- 튜닝: 4 model × 12 × 3 tier × 20 trials ≈ ~60분
- static + expanding: 추가 ~30분
- **총 1.5~2시간 예상**

In [ ]:
results     = []
preds_cache = {}   # (model, regime, country, tier, protocol) → pd.Series
tuned_cache = {}   # (model, regime, country, tier) → (best_params, best_qlike)

# best model 저장 디렉토리
BEST_MODELS_DIR = config.RESULTS_DIR / 'best_models'
for model_name in ALL_DL_MODEL_NAMES:
    (BEST_MODELS_DIR / model_name).mkdir(parents=True, exist_ok=True)

t_total = time.time()

for regime, country in product(config.REGIMES, config.COUNTRIES):
    train, valid, test = load_split_3way(regime, country)
    combined = pd.concat([train, valid])
    print(f'\n[{regime}/{country}] '
          f'train={len(train)} valid={len(valid)} '
          f'combined={len(combined)} test={len(test)}')

    for tier in TIERS:
        feats = get_feature_list(train, country, tier)

        for model_name in ALL_DL_MODEL_NAMES:
            t_case   = time.time()
            n_trials = N_TRIALS_OVERRIDE or DL_N_TRIALS

            # ── 1) Optuna 튜닝 ──
            best_params, best_qlike = tune_dl_model(
                model_name, train, valid, feats,
                n_trials=n_trials, seed=SEED, device=DEVICE,
                **TRAIN_CFG,
            )
            tuned_cache[(model_name, regime, country, tier)] = (best_params, best_qlike)

            # ── 2) 프로토콜별 예측 ──
            for protocol in PROTOCOLS:
                if protocol == 'static':
                    y_pred, final_model = run_dl_static(
                        model_name, combined, test, feats, best_params,
                        seed=SEED, device=DEVICE, **TRAIN_CFG,
                    )
                    # best model 저장 (static만 — expanding은 단일 저장 부적합)
                    pkl_path = (
                        BEST_MODELS_DIR / model_name
                        / f'{regime}_{country}_{tier}.pkl'
                    )
                    joblib.dump({
                        'model_state'   : final_model.cpu().state_dict(),
                        'model_name'    : model_name,
                        'feature_cols'  : feats,
                        'best_params'   : best_params,
                        'best_val_qlike': best_qlike,
                        'regime'        : regime,
                        'country'       : country,
                        'tier'          : tier,
                        'seq_len'       : DL_SEQ_LEN[model_name],
                        'fit_data'      : 'train+valid',
                    }, pkl_path)

                else:  # expanding
                    y_pred = run_dl_expanding(
                        model_name, combined, test, feats, best_params,
                        refit_every=REFIT_EVERY,
                        seed=SEED, device=DEVICE,
                        **EXPAND_CFG,
                    )

                preds_cache[(model_name, regime, country, tier, protocol)] = y_pred

                # ── 3) phase별 평가 ──
                for phase_name, mask, _color in iter_phases(test, regime):
                    if mask.sum() == 0:
                        continue
                    metrics = evaluate(
                        test.loc[mask, 'RV_target'],
                        y_pred[mask],
                    )
                    results.append({
                        'model'         : model_name,
                        'regime'        : regime,
                        'country'       : country,
                        'feature_set'   : tier,
                        'protocol'      : protocol,
                        'phase'         : phase_name,
                        **metrics,
                        'best_val_qlike': best_qlike,
                        'n_features'    : len(feats),
                        'seq_len'       : DL_SEQ_LEN[model_name],
                    })

            elapsed = time.time() - t_case
            print(f'  {tier:<10} {model_name:<8} '
                  f'({elapsed:.1f}s)  best_val_qlike={best_qlike:.4f}')

results_df = pd.DataFrame(results)
print(f'\n총 행: {len(results_df)}  '
      f'/ 총 소요: {(time.time() - t_total) / 60:.1f}분')
print(f'best models 저장 위치: {BEST_MODELS_DIR}')

## 4. 결과 저장 — tier별 분리 CSV

In [ ]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

saved_files = []
for tier in TIERS:
    df_tier = results_df[results_df['feature_set'] == tier].copy()
    out     = config.RESULTS_DIR / f'dl_results_{tier}.csv'
    df_tier.to_csv(out, index=False)
    saved_files.append((out, len(df_tier)))
    print(f'  saved → {out.name}  ({len(df_tier)}행)')

total_rows = sum(n for _, n in saved_files)
print(f'\n총합: {total_rows}행')
print('예상: 4 model × 4 regime × 3 country × 2 protocol × phase수')
results_df.head()


## 5. 예측 로그 저장

- `log/{model}/log_{model}_{regime}_{country}.csv`
- 컬럼: date, feature_set, protocol, RV_target, RV_pred


In [ ]:
LOG_DIR = PROJECT_ROOT / 'log'
LOG_DIR.mkdir(parents=True, exist_ok=True)
for model_name in ALL_DL_MODEL_NAMES:
    (LOG_DIR / model_name).mkdir(parents=True, exist_ok=True)

# (model, regime, country) 단위로 그룹핑
groups: dict = defaultdict(list)
for (model_name, regime, country, tier, protocol), y_pred in preds_cache.items():
    groups[(model_name, regime, country)].append((tier, protocol, y_pred))

saved_logs = []
for (model_name, regime, country), entries in groups.items():
    _, _, test = load_split_3way(regime, country)
    test_idx   = test.index

    rows = []
    for tier, protocol, y_pred in entries:
        rows.append(pd.DataFrame({
            'date'       : test_idx,
            'feature_set': tier,
            'protocol'   : protocol,
            'RV_target'  : test['RV_target'].values,
            'RV_pred'    : y_pred.reindex(test_idx).values,
        }))

    out_df   = pd.concat(rows, ignore_index=True)
    out_path = (
        LOG_DIR / model_name
        / f'log_{model_name}_{regime}_{country}.csv'
    )
    out_df.to_csv(out_path, index=False)
    saved_logs.append(out_path)

print(f'saved → {len(saved_logs)}개 CSV')
print(f'예상: 4 model × 12 (regime × country) = 48 파일')
print(f'예시: {saved_logs[0].relative_to(PROJECT_ROOT)}')

## 6. 시각화 — 위기 phase 음영

In [ ]:
def plot_dl_predictions(
    regime  : str,
    country : str,
    tier    : str = 'extended',
    protocol: str = 'static',
) -> None:
    _, _, test = load_split_3way(regime, country)
    fig, ax = plt.subplots(figsize=(12, 4))

    ax.plot(test.index, test['RV_target'],
            color='black', lw=1.2, label='RV_target')

    for model_name in ALL_DL_MODEL_NAMES:
        key = (model_name, regime, country, tier, protocol)
        if key in preds_cache:
            ax.plot(
                test.index, preds_cache[key], lw=1, alpha=0.75,
                label=f'{model_name}(L={DL_SEQ_LEN[model_name]})',
            )

    for name, mask, color in iter_phases(test, regime):
        if name == 'Full Test' or color is None:
            continue
        idx = test.index[mask]
        if len(idx) > 0:
            ax.axvspan(idx.min(), idx.max(),
                       color=color, alpha=0.12, label=name)

    ax.set_title(f'{regime.upper()} / {country} / {tier} / {protocol}')
    ax.set_ylabel('RV (variance, %²)')
    ax.legend(loc='upper left', fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

for regime in ['gfc', 'covid']:
    for country in config.COUNTRIES:
        plot_dl_predictions(regime, country, 'extended', 'static')

## 7. Tier 효과 분석 — Core → Momentum → Extended

In [ ]:
tier_effect = (
    results_df[results_df['phase'] == 'Full Test']
    .groupby(['feature_set', 'model'])['QLIKE']
    .mean()
    .unstack('feature_set')
    .reindex(columns=TIERS)
    .round(4)
)
tier_effect['Δ Core→Momentum']     = (
    tier_effect['momentum'] - tier_effect['core']
).round(4)
tier_effect['Δ Momentum→Extended'] = (
    tier_effect['extended'] - tier_effect['momentum']
).round(4)
tier_effect

## 8. Tuned hyperparameters 저장

In [ ]:
tuned_rows = []
for (model, regime, country, tier), (params, qlike) in tuned_cache.items():
    row = {
        'model'         : model,
        'regime'        : regime,
        'country'       : country,
        'feature_set'   : tier,
        'best_val_qlike': qlike,
        'seq_len'       : DL_SEQ_LEN[model],
    }
    row.update({f'param_{k}': v for k, v in params.items()})
    tuned_rows.append(row)

tuned_df = pd.DataFrame(tuned_rows)
tuned_df.to_csv(config.RESULTS_DIR / 'dl_tuned_params.csv', index=False)
print(f'saved → dl_tuned_params.csv ({len(tuned_df)}행)')
print(f'예상: 4 model × 4 regime × 3 country × 3 tier = 144행')
tuned_df.head()